In [1]:
# Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Distributed Shared Variables")
    .master("spark://016e64429d30:7077")
    .config("spark.cores.max", 16)
    .config("spark.executor.cores", 4)
    .config("spark.executor.memory", "512M")
    .config("spark.pyspark.python", "python3")
    .getOrCreate()
)

spark

In [2]:
# Read EMP CSV data

_schema = "first_name string, last_name string, job_title string, dob string, email string, phone string, salary double, department_id int"

emp = spark.read.format("csv").schema(_schema).option("header", True).load("/data/input/employee_records.csv")

In [3]:
# Variable (Lookup)
dept_names = {1 : 'Department 1', 
              2 : 'Department 2', 
              3 : 'Department 3', 
              4 : 'Department 4',
              5 : 'Department 5', 
              6 : 'Department 6', 
              7 : 'Department 7', 
              8 : 'Department 8', 
              9 : 'Department 9', 
              10 : 'Department 10'}

In [4]:
# C:\Users\vivek>docker exec -it 57487b82dfa5 bash
# bash-5.0# ls -l /usr/local/bin/python
# ls: cannot access '/usr/local/bin/python': No such file or directory
# bash-5.0# mkdir -p /usr/local/bin
# bash-5.0# ln -s /usr/bin/python3 /usr/local/bin/python
# bash-5.0# ls -l /usr/local/bin/python
# lrwxrwxrwx 1 root root 16 Apr  9 20:14 /usr/local/bin/python -> /usr/bin/python3

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# 1. Define a UDF that maps department_id to department name
def get_department_name(dept_id):
    return dept_names.get(dept_id, "Unknown")  # default to "Unknown" if not found

dept_udf = udf(get_department_name, StringType())

# 2. Add new column 'department' using the UDF
emp_with_dept = emp.withColumn("department", dept_udf(emp["department_id"]))

# 3. Show result
emp_with_dept.show()

# Problem:
# The lookup dictionary is serialized and sent with every task
# Causes:
# High network overhead
# Repeated data transfer
# Poor scalability
# This is inefficient for large clusters.

+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+-------------+
|first_name| last_name|           job_title|       dob|               email|               phone|  salary|department_id|   department|
+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+-------------+
|   Richard|  Morrison|Public relations ...|1973-05-05|melissagarcia@exa...|       (699)525-4827|512653.0|            8| Department 8|
|     Bobby|  Mccarthy|   Barrister's clerk|1974-04-25|   llara@example.net|  (750)846-1602x7458|999836.0|            7| Department 7|
|    Dennis|    Norman|Land/geomatics su...|1990-06-24| jturner@example.net|    873.820.0518x825|131900.0|           10|Department 10|
|      John|    Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506.0|            1| Department 1|
|  Michelle|   Elliott|      Air cabin crew|1975-03-31|

In [5]:
# Broadcast the variable

broadcast_dept_names = spark.sparkContext.broadcast(dept_names)

# Lookup data is sent once per executor
# Much lower network usage

In [7]:
# Check the value of the variable

type(broadcast_dept_names)

broadcast_dept_names.value

{1: 'Department 1',
 2: 'Department 2',
 3: 'Department 3',
 4: 'Department 4',
 5: 'Department 5',
 6: 'Department 6',
 7: 'Department 7',
 8: 'Department 8',
 9: 'Department 9',
 10: 'Department 10'}

In [8]:
# Create UDF to return Department name

from pyspark.sql.functions import udf, col

@udf
def get_dept_names(dept_id):
    return broadcast_dept_names.value.get(dept_id)

In [9]:
emp_final = emp.withColumn("dept_name", get_dept_names(col("department_id")))

In [10]:
emp_final.show()

+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+-------------+
|first_name| last_name|           job_title|       dob|               email|               phone|  salary|department_id|    dept_name|
+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+-------------+
|   Richard|  Morrison|Public relations ...|1973-05-05|melissagarcia@exa...|       (699)525-4827|512653.0|            8| Department 8|
|     Bobby|  Mccarthy|   Barrister's clerk|1974-04-25|   llara@example.net|  (750)846-1602x7458|999836.0|            7| Department 7|
|    Dennis|    Norman|Land/geomatics su...|1990-06-24| jturner@example.net|    873.820.0518x825|131900.0|           10|Department 10|
|      John|    Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506.0|            1| Department 1|
|  Michelle|   Elliott|      Air cabin crew|1975-03-31|

In [11]:
# Calculate total salary of Department 6

from pyspark.sql.functions import sum

emp.where("department_id = 6").groupBy("department_id").agg(sum("salary").cast("long")).show()

+-------------+---------------------------+
|department_id|CAST(sum(salary) AS BIGINT)|
+-------------+---------------------------+
|            6|                50294510721|
+-------------+---------------------------+



In [12]:
# Another way to execute above task is by using Accumulators
# Accumulators are variables in Spark used for aggregating information across executors.
# They are write-only for workers (executors) but readable on the driver.

dept_sal = spark.sparkContext.accumulator(0)

# Creates a distributed variable to sum salaries across workers.
# Initially 0.0.
# Important: executors can add() to it, driver can read .value, but workers cannot see updates from other tasks.
# Using a float (0.0) is necessary for salary which is a double.

In [13]:
# Use foreach

def calculate_salary(department_id, salary):
    if department_id == 6:
        dept_sal.add(salary)

emp.foreach(lambda row : calculate_salary(row.department_id, row.salary))

# foreach executes the function on every partition in parallel across workers.
# For every row, the lambda calls calculate_salary(row.department_id, row.salary).
# Each worker updates its local copy of the accumulator, and Spark merges these updates back to the driver.

In [14]:
# View total value

dept_sal.value

50294510721.0

In [15]:
spark.stop()